# What is actually in the data?

A reference inventory of every data source on disk. For each directory: how many files,
how big, and — from one example file — the columns, the dtypes, the row count, and a few
rows. Column meanings are quoted from the source that produced them, with line numbers, so
each one can be checked.

This notebook shows what is there. It does not draw conclusions about it, and **it does not
modify or write any file** — every cell reads.

Shared helpers come from [`ctabus.py`](ctabus.py), which
[`ten_minute_promise.ipynb`](ten_minute_promise.ipynb) imports too, so the palette and the
loaders cannot drift apart between the two.

### Contents

- **§1 — Census.** What is on disk: file counts, sizes, dates.
- **§2 — One example file per source.** Columns, dtypes, sample rows, and what the columns mean.
- **§3 — Route geometry.** Four sources of CTA bus route shapes, and how they differ.
- **§4 — Looking at any route's stops.** Stops, `pid` and `stop_sequence` plotted, for a
  route you choose.

### Dates

Three dates are tracked per source where knowable: when the data covers, when the publisher
last updated it, and when we downloaded it. The download date is recorded for repeatability,
and because which routes and buses appear in a file depends on when it was pulled.


In [ ]:
import glob
import itertools
import json
import os
import zipfile
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import geopandas as gpd
import matplotlib.pyplot as plt

# Everything shared with ten_minute_promise.ipynb lives in ctabus.py: the
# palette, the paths into data/, the file-inspection helpers, and the loaders
# for route shapes and stop locations. Kept in one place so the two notebooks
# cannot drift apart.
import ctabus as cta
from ctabus import (SURFACE, INK, INK2, MUTED, GRID, AXIS,
                    BLUE, ORANGE, AQUA, BLUE_L,
                    DATA, STOPWATCH, GEO, GTFS, DERIVED,
                    human, peek, census, style)

cta.apply_style()
pd.set_option('display.max_columns', 60, 'display.width', 200)

ROUTE = '66'      # the route ten_minute_promise.ipynb examines; §4 uses it as its example

print('geopandas', gpd.__version__, '| pandas', pd.__version__, '| pyarrow', pa.__version__)
print('data ->', os.path.realpath(DATA))


## 1. Census — what is on disk

`data/` is a symlink out of the repo to `/media/work/data/cta`, so none of this is version
controlled. The census below is the check that a working copy has what the notebooks expect.


In [ ]:
inventory = cta.census(DATA)
print(f'total on disk: {human(inventory.bytes.sum())}   across {inventory.files.sum():,} files\n')
display(inventory.drop(columns='bytes'))


## 2. One example file from each source

`peek()` reports the row count, every column with its dtype, and the first few rows. For
parquet it reads the schema and a single row group, so the 32 GB directory is never loaded.

### Where each source came from, and how to get it again

| Path | What it is | Where it came from | Re-fetch with |
|---|---|---|---|
| `cta_bus_daily.csv` | boardings by route × day, 2001-01-01 → 2026-05-31, 188 routes | Chicago Data Portal [`jyb9-n7fm`](https://data.cityofchicago.org/Transportation/CTA-Ridership-Bus-Routes-Daily-Totals-by-Route/jyb9-n7fm/about_data) | `curl` — [`README.md`](README.md) §Data |
| `cta_bus_monthly.csv` | monthly day-type averages; the only source of route *names* | Portal [`bynn-gwxy`](https://data.cityofchicago.org/Transportation/CTA-Ridership-Bus-Routes-Monthly-Day-Type-Averages/bynn-gwxy) | `curl` — [`README.md`](README.md) §Data |
| `rt_to_pid.csv` | route → pattern crosswalk, ~940 rows | StopWatch CDN | `fetch_stopwatch.py` (`read_xwalk`) |
| `stopwatch/processed_by_pid/` | **actual** arrivals: one row per bus per stop, interpolated from pings | [Mansueto StopWatch](https://github.com/mansueto-institute/cta-stop-watch) | `fetch_stopwatch.py --what processed` |
| `stopwatch/clean_timetables/` | **scheduled** arrivals, same shape, from CTA's own GTFS | StopWatch | `fetch_stopwatch.py --what timetables` |
| `stopwatch/metrics/` | StopWatch's own aggregation (headway/trip-time quantiles) | StopWatch | `fetch_stopwatch.py --what metrics` |
| `stopwatch/full_day_data/` | **raw** unprocessed 5-minute Bus Tracker pings | StopWatch | `fetch_stopwatch.py --what raw --start … --end …` |
| `geo/` | route line geometry — four sources, compared in §3 | Portal `6uva-a5ei`, `atza-xq2n`, `d5bx-dr8z` | §3 below; `exploration.ipynb` §4.a |
| `gtfs/` | CTA schedule feeds, 4 dated archives | **unknown** — see note below | — |
| `derived/` | written by this repo's own notebooks, not downloaded | `exploration.ipynb`, `seasonality.ipynb`, `ten_minute_promise.ipynb` | re-run those notebooks |

**On the StopWatch archive.** It continues the [Chi Hack Night Ghost Buses](https://github.com/chihacknight/chn-ghost-buses)
scrape: CTA Bus Tracker `getvehicles` polled every 5 minutes since 2022-05-19, then
interpolated onto each route's fixed path to estimate when each bus passed each stop. It is
the only public source of *realised* frequency — the ridership files carry no service measure
at all. `bus_stop_time` in `processed_by_pid` is therefore **estimated, not observed**.
StopWatch's own published validation covers June 2022 – July 2024 only.

**Gap: we do not know where `gtfs/` came from.** The four zips are named
`cta_gtfs_<timestamp>.zip` and contain no `feed_info.txt`, so nothing in the files says who
published them or how they were retrieved. `agency.txt` confirms only that the agency is CTA.
`README.md` documents every other source but not this one. Treat the timestamps in the
filenames as download dates of unknown origin until someone works out where they came from.


### 2a. `processed_by_pid/` — actual arrivals

One row per bus per stop.

**Column meanings, read out of the StopWatch source.** None of these are documented in
StopWatch's README; the definitions below come from
[`report_automation/interpolation.py`](https://github.com/mansueto-institute/cta-stop-watch/blob/main/cta-stop-watch/report_automation/interpolation.py)
and
[`calculate_stop_time.py`](https://github.com/mansueto-institute/cta-stop-watch/blob/main/cta-stop-watch/report_automation/calculate_stop_time.py),
line numbers as of 2026-08-12.

| Column | Meaning | Source |
|---|---|---|
| `bus_stop_time` | **estimated** time the bus passed the stop — linear interpolation *in distance* between the two pings that bracket it. Naive local time, no timezone. | `interpolation.py:70-74` |
| `stpid` | CTA stop id | |
| `p_stp_id` | pattern-stop id (stop as it appears within one pattern) | |
| `stop_sequence` | position along the pattern; a running count of stop rows | `interpolation.py:21` |
| `pid` | pattern id — one specific path along the route | |
| `rt` | route; constant within a file | `calculate_stop_time.py:224` |
| `vid` | vehicle id | `calculate_stop_time.py:222` |
| `unique_trip_vehicle_day` | trip key, assigned from the grouping key in `process_one_trip` | `calculate_stop_time.py:218` |
| `typ` | row kind: `B` = vehicle ping, `S` = stop. Output filtered to `S` only | `interpolation.py:167` |
| `seg_combined` | pattern segment index; renamed from `segment`. Construction not traced. | `calculate_stop_time.py:120` |
| `speed_mph` | average speed over the ping-to-ping segment, broadcast to every stop in it | `interpolation.py:32`, `:81` |

**How the times are built.** The interpolation quoted at `:70-74`:

```python
stops_df["bus_stop_time"] = stops_df["data_time_y"] + (
    stops_df["ping_time_diff"]
    * stops_df["accumulated_distance"]
    / stops_df["ping_dist"].replace(0, 0)
)
```

Distances are metres — the frame is reprojected to `epsg:26971` (NAD83 Illinois East) at
`:12`. Note `.replace(0, 0)` substitutes zero for zero, so it does not guard the division it
sits in.

Stops falling before a trip's first ping or after its last are **extrapolated** rather than
interpolated (`:145-155`), stepping outward by `stop_dist / (speed_mph / 2.23694)`.

`speed_mph` is post-processed at `:110-115`:

```python
# replace values below 1 or above 115 for speed_mph
stops_df["speed_mph"] = stops_df["speed_mph"].apply(
    lambda x: np.nan if x < 1 or x > 115 else x)
stops_df["speed_mph"] = stops_df["speed_mph"].fillna(method="ffill")
stops_df["speed_mph"] = stops_df["speed_mph"].fillna(method="bfill")
```

§2c plots the column.


In [ ]:
_ = peek(f'{STOPWATCH}/processed_by_pid/trips_100_full.parquet',
         note='one pattern of route 100; 886 such files, one per pattern')


### 2b. The other StopWatch products, GTFS, and the ridership files

`clean_timetables/` is the **scheduled** counterpart to `processed_by_pid/` — same shape, one
row per stop visit with a `bus_stop_time`. Two naming differences when joining them, already
recorded in [`docs/bus-tracker-data-plan.md`](docs/bus-tracker-data-plan.md): the stop id is
`stpid` in actuals but `stop_id` in schedules, and `pid` is zero-padded (`06672`) in schedules
but float-like (`20426.0`) in actuals.

`metrics/stop_metrics_df_latest.parquet` is StopWatch's own aggregation. How its headway
column is built, from
[`stop_metrics.py:19-24`](https://github.com/mansueto-institute/cta-stop-watch/blob/main/cta-stop-watch/report_automation/stop_metrics.py):

```python
trips_df = trips_df.sort(["stop_id", "bus_stop_time"])
trips_df = trips_df.with_columns(
    time_till_next_bus=(
        pl.col("bus_stop_time").shift(-1) - pl.col("bus_stop_time")
    ).over(pl.col("stop_id").rle_id()))
```

The grouping key is `stop_id` alone — not direction, and not date. Units are **seconds**
(`stop_metrics.py:36-37`). The `is_daytime` filter is `dt.hour().is_between(6, 20)`, inclusive
at both ends, i.e. 06:00–20:59.

`full_day_data/` is the raw 5-minute ping feed the interpolation consumes.


In [ ]:
_ = peek(f'{STOPWATCH}/clean_timetables/rt100_timetable.parquet',
         note='SCHEDULED arrivals for route 100 — the comparison series')
print()
_ = peek(f'{STOPWATCH}/metrics/stop_metrics_df_latest.parquet',
         note='StopWatch aggregation. Units are SECONDS. Not used by the analysis (see above).')


In [ ]:
raw_days = sorted(glob.glob(f'{STOPWATCH}/full_day_data/*.csv'))
print(f'raw ping days on disk: {len(raw_days)}  '
      f'({os.path.basename(raw_days[0])[:10]} … {os.path.basename(raw_days[-1])[:10]})\n')
_ = peek(raw_days[0], note='RAW 5-minute Bus Tracker pings — the input to the interpolation')


In [ ]:
_ = peek(f'{DATA}/cta_bus_daily.csv',
         note='ridership: boardings by route x day. NOTE the 2025 restatement — '
              'docs/ridership-restatement-notes.md')
print()
_ = peek(f'{DATA}/cta_bus_monthly.csv', note='ridership monthly; the only source of route names')
print()
_ = peek(f'{DATA}/rt_to_pid.csv', note='route -> pattern crosswalk used by fetch_stopwatch.py')


In [ ]:
import hashlib

# The zip NAME carries one timestamp; the files INSIDE carry another. Both are shown,
# alongside our download date (the file mtime).
#
# Only .txt members are timestamped: developers_license_agreement.htm is dated 2014-07-30
# in every archive and would otherwise dominate the minimum.
rows = []
for z in sorted(glob.glob(f'{GTFS}/*.zip')):
    with zipfile.ZipFile(z) as zf:
        dates = [datetime(*i.date_time) for i in zf.infolist() if i.filename.endswith('.txt')]
        digest = hashlib.md5(zf.read('stops.txt')).hexdigest()
    stamp = os.path.basename(z).replace('cta_gtfs_', '').replace('.zip', '')
    rows.append({
        'zip': os.path.basename(z),
        'name_timestamp': f'{stamp[:4]}-{stamp[4:6]}-{stamp[6:8]} {stamp[8:10]}:{stamp[10:12]}',
        'txt_earliest': min(dates).strftime('%Y-%m-%d'),
        'txt_latest': max(dates).strftime('%Y-%m-%d'),
        'downloaded': datetime.fromtimestamp(os.path.getmtime(z)).strftime('%Y-%m-%d'),
        'size': human(os.path.getsize(z)),
        'stops.txt md5': digest[:12],
    })
gtfs_zips = pd.DataFrame(rows)

loose_md5 = hashlib.md5(open(f'{GTFS}/stops.txt', 'rb').read()).hexdigest()
gtfs_zips['== loose stops.txt'] = gtfs_zips['stops.txt md5'] == loose_md5[:12]
display(gtfs_zips)

match = gtfs_zips.loc[gtfs_zips['== loose stops.txt'], 'zip']
print(f'loose .txt files in {GTFS}/ match the stops.txt inside: '
      f'{match.iloc[0] if len(match) else "no zip here"}')
print('\nStopWatch sources its own historic GTFS from Transit.land (upstream README: '
      '"Historic feeds back to May 2022 were downloaded"). Whether these four zips came the')
print('same way is not recorded anywhere in this repo or in the files.')


### 2c. What `speed_mph` looks like

The distribution, the two ends of its range, and one trip's values along the route.

For reference while reading these, the construction in `interpolation.py` is: one speed per
ping-to-ping segment (`:32`, `:81`), values below 1 or above 115 replaced with `NaN` and then
`ffill`/`bfill`ed from neighbouring rows (`:110-115`).


In [ ]:
# Pick a file to look at speed_mph in. cta.pattern_files() also reports which patterns
# the crosswalk lists but the server does not have (54 of 940 overall).
paths, missing_pids = cta.pattern_files(ROUTE)
print(f'route {ROUTE}: {len(paths) + len(missing_pids)} patterns in the crosswalk, '
      f'{len(paths)} with a file, {len(missing_pids)} missing')
print(f'missing pids: {missing_pids}')

# The largest file, so the distribution is not dominated by one thin pattern.
biggest_path = max(paths, key=os.path.getsize)
print(f'\nusing: {biggest_path}  ({human(os.path.getsize(biggest_path))})')


In [ ]:
speed_sample = pq.read_table(
    biggest_path,
    columns=['speed_mph', 'typ', 'stop_sequence', 'bus_stop_time',
             'unique_trip_vehicle_day']).to_pandas()

s = speed_sample.speed_mph
print(f'rows        : {len(speed_sample):,}      nulls: {s.isna().sum():,}')
print(f'min / max   : {s.min():.6f} / {s.max():.4f}')
print(f'below 1 mph : {(s < 1).sum():,}       above 115 mph: {(s > 115).sum():,}')
print(f'exactly 1.0 : {(s == 1).sum():,}')
print(f'typ values  : {speed_sample.typ.value_counts().to_dict()}')
print()
print(s.describe([.01, .05, .25, .5, .75, .95, .99]).round(3).to_string())

# How much does speed vary within a single trip?
speed_sample = speed_sample.sort_values(['unique_trip_vehicle_day', 'stop_sequence'])
grp = speed_sample.groupby('unique_trip_vehicle_day', observed=True)
print(f'\nconsecutive stops in a trip with an identical speed: {grp.speed_mph.diff().eq(0).mean():.1%}')
print(f'distinct speeds per trip (median): {grp.speed_mph.nunique().median():.0f}')
print(f'stops per trip (median)          : {grp.size().median():.0f}')


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.4))
# Dashed orange lines mark the 1 and 115 mph bounds from interpolation.py:110-113.

ax = axes[0]
ax.hist(s.dropna(), bins=120, color=BLUE, edgecolor='none')
ax.axvline(1, color=ORANGE, lw=1.2, ls='--')
ax.axvline(115, color=ORANGE, lw=1.2, ls='--')
ax.set_title('all values')
ax.set_xlabel('speed_mph')
style(ax, 'stop visits')

ax = axes[1]
ax.hist(s[s < 12].dropna(), bins=np.arange(0, 12.25, 0.25), color=BLUE, edgecolor='none')
ax.axvline(1, color=ORANGE, lw=1.2, ls='--')
ax.set_title('zoom: 0-12 mph')
ax.set_xlabel('speed_mph')
style(ax, 'stop visits')

ax = axes[2]
ax.hist(s[s > 20].dropna(), bins=40, color=BLUE, edgecolor='none')
ax.set_title(f'above 20 mph  (max {s.max():.1f})')
ax.set_xlabel('speed_mph')
style(ax, 'stop visits')

# One trip's speed against position along the route.
ax = axes[3]
example_trip = grp.size().sort_values().index[len(grp) // 2]      # a median-length trip
one = speed_sample[speed_sample.unique_trip_vehicle_day == example_trip]
ax.step(one.stop_sequence, one.speed_mph, where='post', color=BLUE, lw=1.4)
ax.set_title('one trip, stop by stop')
ax.set_xlabel('stop_sequence')
style(ax, 'speed_mph')

plt.tight_layout()
plt.show()


## 3. Route geometry — four sources

Four files on disk describe the shape of CTA bus routes:

| File | Portal dataset | Downloaded |
|---|---|---|
| `cta_routes_current.geojson` | [`6uva-a5ei`](https://data.cityofchicago.org/Transportation/CTA-Bus-Routes/6uva-a5ei/about_data) — "CTA - Bus Routes" | 2026-08-03 |
| `portal_6uva-a5ei_busroutes.geojson` | the same dataset, pulled again | 2026-08-12 |
| `cta_routes_2015.kml` | [`atza-xq2n`](https://data.cityofchicago.org/Transportation/CTA-Bus-Routes-KML-Deprecated-February-2015-/atza-xq2n) — "CTA - Bus Routes - KML (Deprecated February 2015)" | 2026-08-03 |
| `d5bx-dr8z_shapefile/CTA_BusRoutes.shp` | [`d5bx-dr8z`](https://data.cityofchicago.org/Transportation/CTA-Bus-Routes-Shapefile/d5bx-dr8z/about_data) — "CTA - Bus Routes - Shapefile" | 2026-08-12 |

The two 2026-08-12 files are the ones fetched for this notebook. The portal's own metadata for
each was saved next to them as `portal_<id>_metadata.json`, and is read below rather than
retyped.

The cells that follow report, for each source: feature count, CRS, attribute columns, the set
of route ids, and — for pairs that overlap — whether the geometries are identical.


In [ ]:
# What the portal itself says about the two datasets we fetched today.
meta_rows = []
for meta_path in sorted(glob.glob(f'{GEO}/portal_*_metadata.json')):
    m = json.load(open(meta_path))
    to_date = lambda v: (datetime.fromtimestamp(v, timezone.utc).strftime('%Y-%m-%d')
                         if isinstance(v, int) else None)
    meta_rows.append({
        'dataset': m.get('id'),
        'name': m.get('name'),
        'displayType': m.get('displayType'),
        'createdAt': to_date(m.get('createdAt')),
        'rowsUpdatedAt': to_date(m.get('rowsUpdatedAt')),
        'viewLastModified': to_date(m.get('viewLastModified')),
    })
display(pd.DataFrame(meta_rows))

print('rowsUpdatedAt is when the DATA last changed; viewLastModified is when the portal PAGE')
print('was last edited. They are different fields and can be years apart.')


In [ ]:
# cta.load_geometry() reads all four files and adds a `route_id` column to each,
# so they can be compared despite naming the route column differently.
geo_sources = cta.GEO_SOURCES
geo = cta.load_geometry()

rows = [{
    'source': key,
    'features': len(g),
    'distinct routes': g.route_id.nunique(),
    'CRS': str(g.crs),
    'geom types': '/'.join(sorted(g.geometry.geom_type.unique())),
    'attribute columns': ', '.join(c for c in g.columns if c not in ('geometry', 'route_id')),
} for key, g in geo.items()]

summary = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', 90)
display(summary)


In [ ]:
route_sets = {k: set(g.route_id) for k, g in geo.items()}

# Which routes appear in which source.
all_routes = sorted(set().union(*route_sets.values()),
                    key=lambda r: (not r[:1].isdigit(), r.zfill(4)))
membership = pd.DataFrame(
    {k: [r in s for r in all_routes] for k, s in route_sets.items()}, index=all_routes)
membership.index.name = 'route'

print(f'{len(all_routes)} distinct route ids across all four sources\n')
for a, b in itertools.combinations(route_sets, 2):
    only_a, only_b = route_sets[a] - route_sets[b], route_sets[b] - route_sets[a]
    print(f'{a:22} vs {b:22} shared {len(route_sets[a] & route_sets[b]):>4}   '
          f'only-left {len(only_a):>3}   only-right {len(only_b):>3}')

print('\nroutes NOT in every source:')
partial = membership[~membership.all(axis=1)]
display(partial)


In [ ]:
# Are the geometries the same, for routes a pair has in common? Everything is put in
# EPSG:4326 first, then compared with a 0.1 m tolerance so that coordinate rounding in
# different file formats does not read as a real difference.
def compare_geometry(a, b, tol=0.1):
    ga, gb = geo[a].to_crs(4326), geo[b].to_crs(4326)
    ga = ga.dissolve('route_id').geometry          # one geometry per route
    gb = gb.dissolve('route_id').geometry
    shared = sorted(set(ga.index) & set(gb.index))
    if not shared:
        return None
    ga, gb = ga.loc[shared], gb.loc[shared]
    # metres: compare in the projected CRS the shapefile ships in
    ga_m = gpd.GeoSeries(ga, crs=4326).to_crs(3435)
    gb_m = gpd.GeoSeries(gb, crs=4326).to_crs(3435)
    equal = ga_m.geom_equals_exact(gb_m, tolerance=tol)
    hausdorff = ga_m.hausdorff_distance(gb_m, align=True)
    return pd.DataFrame({
        'routes compared': [len(shared)],
        f'identical (<{tol} m)': [int(equal.sum())],
        'differ': [int((~equal).sum())],
        'median max-deviation (ft)': [round(hausdorff.median(), 1)],
        'largest max-deviation (ft)': [round(hausdorff.max(), 1)],
    }, index=[f'{a}  vs  {b}'])

comparisons = [compare_geometry(a, b) for a, b in itertools.combinations(geo, 2)]
display(pd.concat([c for c in comparisons if c is not None]))
print('EPSG:3435 is NAD83 Illinois East in US survey feet, so deviations are in feet.')


In [ ]:
# Per-route deviation between the current GeoJSON and the d5bx-dr8z shapefile.
a = geo['geojson_portal_0812'].dissolve('route_id').geometry.to_crs(3435)
b = geo['shp_d5bx-dr8z'].dissolve('route_id').geometry.to_crs(3435)
shared = sorted(set(a.index) & set(b.index))
dev = pd.DataFrame({
    'max_deviation_ft': a.loc[shared].hausdorff_distance(b.loc[shared], align=True).round(1),
    'geojson_len_mi': (a.loc[shared].length / 5280).round(2),
    'shapefile_len_mi': (b.loc[shared].length / 5280).round(2),
})
dev['len_diff_mi'] = (dev.shapefile_len_mi - dev.geojson_len_mi).round(2)
print('routes where the two disagree by more than 1 ft:')
display(dev[dev.max_deviation_ft > 1].sort_values('max_deviation_ft', ascending=False))

print(f'\nroutes agreeing to within 1 ft: {(dev.max_deviation_ft <= 1).sum()} of {len(dev)}')


In [ ]:
# All four files are kept on disk. This cell reports how much content they share.
# Two measures, because they disagree here:
#   - "same route set?"      -- do the files cover the same routes at all
#   - "routes agreeing"      -- of the shared routes, how many match within 1 ft
TOL_FT = 1.0

def agreement(a, b, tol_ft=TOL_FT):
    ga = geo[a].dissolve('route_id').geometry.to_crs(3435)
    gb = geo[b].dissolve('route_id').geometry.to_crs(3435)
    idx = sorted(set(ga.index) & set(gb.index))
    d = ga.loc[idx].hausdorff_distance(gb.loc[idx], align=True)
    return {
        'pair': f'{a}  vs  {b}',
        'same route set': route_sets[a] == route_sets[b],
        'routes shared': len(idx),
        f'agree within {tol_ft:g} ft': int((d <= tol_ft).sum()),
        'disagree': int((d > tol_ft).sum()),
        'largest deviation (ft)': round(d.max(), 1),
    }

pairs = pd.DataFrame([agreement(a, b) for a, b in itertools.combinations(geo, 2)])
display(pairs)

file_of = {k: geo_sources[k] for k in geo}
overview = pd.DataFrame([{
    'source': k,
    'features': len(geo[k]),
    'routes': geo[k].route_id.nunique(),
    'CRS': str(geo[k].crs),
    'file size': human(os.path.getsize(file_of[k])),
    'downloaded': datetime.fromtimestamp(os.path.getmtime(file_of[k])).strftime('%Y-%m-%d'),
    'path': file_of[k],
} for k in geo])
display(overview)

print('All four files are kept. The table above is the record of what each one duplicates:')
print('read "routes shared" against "agree within 1 ft" to see how much of a pair is the')
print('same data. The per-route breakdown for the closest pair is in the previous cell.')


## 4. Looking at any route's stops

How to see where a route's stops are and how its patterns lay out along the street. The
route below is a parameter — set `EXAMPLE_ROUTE` to any of the 20 Frequent Network routes,
or any other route in the crosswalk.

`processed_by_pid/` carries no coordinates, only `stpid`. Coordinates come from GTFS
`stops.txt`, joined on the stop id. The join never matches everything, because the GTFS feed
is one snapshot while the arrivals run from 2022, so the match rate is printed before
anything is plotted.

The three plotting helpers live in [`ctabus.py`](ctabus.py):

- `plot_route_overview` — all stops on the route line, ends labelled.
- `plot_pattern_panels` — one small map per pattern, shaded by `stop_sequence`.
- `plot_sequence_vs_longitude` — stop order against longitude, which separates the two
  directions for an east–west route without using any direction label.

Route 66 specifically is examined in
[`ten_minute_promise.ipynb`](ten_minute_promise.ipynb), which is the notebook about that
route. Nothing is written to disk here.


In [ ]:
EXAMPLE_ROUTE = '152'          # change this to look at a different route

paths, missing_pids = cta.pattern_files(EXAMPLE_ROUTE)
print(f'route {EXAMPLE_ROUTE}: {len(paths) + len(missing_pids)} patterns in the crosswalk, '
      f'{len(paths)} with a file, {len(missing_pids)} missing')
print(f'missing pids: {missing_pids}')

example_stops = cta.route_stops(EXAMPLE_ROUTE)
matched = example_stops.stop_lat.notna()
print(f'\npattern-stop rows        : {len(example_stops):,}')
print(f'distinct patterns        : {example_stops.pid.nunique()}')
print(f'distinct stops           : {example_stops.stpid.nunique()}')
print(f'rows with GTFS coords    : {matched.sum():,} ({matched.mean():.1%})')
print(f'distinct stpid unmatched : {example_stops.loc[~matched, "stpid"].nunique()}')
if (~matched).any():
    print(f'unmatched examples       : '
          f'{sorted(example_stops.loc[~matched, "stpid"].unique())[:12]}')

print('\nstops per pattern:')
print(example_stops.groupby('pid').stpid.nunique().sort_values(ascending=False).to_string())
display(example_stops.head(6))


In [ ]:
cta.plot_route_overview(example_stops, EXAMPLE_ROUTE)
plt.tight_layout()
plt.show()


In [ ]:
cta.plot_pattern_panels(example_stops, EXAMPLE_ROUTE)
plt.show()


In [ ]:
cta.plot_sequence_vs_longitude(example_stops, EXAMPLE_ROUTE)
plt.tight_layout()
plt.show()
